# SARIMAX nacional — validação final

A modelagem SARIMAX é a terceira etapa do TCC *Mudanças Climáticas e Inadimplência*. Este notebook complementa Granger e DLM, sem assumir causalidade.

**Pergunta:** os desastres climáticos acrescentam informação à evolução temporal da inadimplência nacional após controlar sua própria dinâmica? A unidade desta etapa é **Brasil × mês**, janeiro/2013–dezembro/2024.

**Entradas versionadas:** `outputs/tables/segunda_etapa/series_nacionais_agregadas.csv`. Bases originais BACEN/SCR e Atlas não foram versionadas; não é possível reconstruir diretamente os grupos COBRADE exatos nem validar novamente os registros brutos.

## 1. Preparação, variável resposta e estacionariedade

A taxa nacional é (I_t=100(sum_i CI_{it})/(sum_i CA_{it})), calculada em etapa anterior. Aplicamos (y_t=100Deltalog(I_t)), e, para cada contagem de desastre, (x_t=Deltalog(1+D_t)). As diferenças são externas ao modelo: usamos (d=D=0) para não diferenciar duas vezes.

O ADF testa a hipótese nula de raiz unitária e o KPSS testa a hipótese nula de estacionariedade. Repete-se a avaliação em série nacional completa e pré-pandemia, sem transferir automaticamente resultados das 27 UFs.

## 2. Identificação e diagnóstico do modelo de referência

Os modelos inicialmente selecionados por BIC ainda apresentavam autocorrelação residual. Uma auditoria documentada em `src/sarimax_diagnostico.py` e `src/sarimax_diagnostico_lags.py` identificou estrutura persistente nos lags 3 e 6. Por esse motivo, foram definidos dois candidatos **parcimoniosos** para o total: SARIMA com AR esparso nos lags 1, 3, 6 e AR sazonal anual (12); e AR(1) com AR semestral (6) e dois harmônicos anuais determinísticos. Para o pré-pandemia, com amostra menor, a comparação se restringe a duas estruturas de poucos parâmetros.

A seleção utiliza exclusivamente a amostra de treinamento (até dezembro/2022), escolhendo o menor BIC entre modelos com Ljung–Box ajustado por graus de liberdade não significativo em 12 e 24 meses. Não rejeição não prova adequação geral; ARCH e normalidade também são documentados.

## 3. Exposições climáticas, defasagens e suporte

Com o modelo-base escolhido, acrescentamos um regressor climático por vez: total de protocolos COBRADE 12/13/14 e tipologias disponíveis na série auditada. São avaliados os lags 0, 1 e 3 da exposição transformada, com **a mesma parte ARMA e mesmos controles determinísticos** para o SARIMA e o SARIMAX. A escolha de lag usa BIC no treino e verifica resíduos. Categorias com menos de 18 meses positivos ou menos de 30 registros no treino são classificadas como insuficientes.

Esta etapa não reconstrói os grupos naturais exatos a partir das colunas administrativas do Atlas: isso depende dos registros brutos. Não se somam classificações sobrepostas.

## 4. Avaliação temporal e inferência

A janela de teste 2023–2024 é reservada para 24 previsões de um mês à frente. A cada observação realizada, apenas o estado do modelo é atualizado, sem reajustar parâmetros. Para lag zero, a previsão é **condicional à exposição contemporânea observada**, não uma previsão operacional que antecipe desastres futuros. Compara-se MAE e RMSE de (100Deltalog(I)).

As estimativas finais dos coeficientes são reajustadas sobre a série completa, após avaliar previsões. IC95%, p-valores e q Benjamini–Hochberg são exploratórios: não incorporam a incerteza da seleção do lag ou da estrutura ARMA. Associação não implica causalidade.

In [ ]:
from pathlib import Path
import runpy

raiz = Path.cwd()
if not (raiz / 'src' / 'sarimax_validacao_final.py').exists():
    raiz = raiz.parent
assert (raiz / 'src' / 'sarimax_validacao_final.py').exists()
runpy.run_path(str(raiz / 'src' / 'sarimax_validacao_final.py'), run_name='__main__')

## 5. Tabelas e interpretação

Primeiro, verifique `baselines.csv` (ordens escolhidas e adequação dos resíduos). Em seguida, compare `modelos.csv` (diferença de BIC) e `metricas.csv` (MAE e RMSE). Os coeficientes e q-valores estão em `coeficientes.csv`, e as verificações residuais em `diagnosticos.csv`. O relatório HTML inclui os principais resultados, sua fundamentação e os limites da interpretação.

**Regra de conclusão:** a inclusão do total climático somente melhora o modelo quanto à métrica em que apresente valor menor, sem generalizar uma melhora pontual como superioridade estatística. Interprete os critérios separadamente e destaque discordâncias.

In [ ]:
import pandas as pd
from IPython.display import display

pasta = raiz / 'outputs/tables/sarimax_final'
for nome in ['baselines','modelos','coeficientes','diagnosticos','metricas','sensibilidade','estacionariedade']:
    print('\n', nome.upper())
    display(pd.read_csv(pasta / f'{nome}.csv').round(4))
print('Relatório HTML:', raiz / 'outputs/reports/relatorio_sarimax_final.html')

## 6. Limitações e integração com as etapas anteriores

O SARIMAX nacional investiga uma associação temporal agregada e avalia previsão condicional. Granger avalia precedência preditiva; o DLM mede relações contemporâneas e defasadas no painel UF×mês. Nenhum dos três métodos permite interpretar automaticamente as regressoras como efeitos causais. O pré-pandemia apresenta ADF/KPSS divergentes para a diferença logarítmica da inadimplência, exigindo cautela. O Atlas bruto é necessário para desagregação COBRADE por grupos sem mistura administrativa.

Consulte `docs/sarimax_resultados_finais.md` e `outputs/reports/relatorio_sarimax_final.html` para a redação acadêmica e os resultados finais.